# Aula MLOps — do gold ao endpoint: Experiments, Model Registry e predição online

Nas aulas anteriores vocês montaram a tubulação e chegaram à camada **gold** de anúncios de imóveis no
BigQuery. Este notebook pega essa gold e leva um modelo até o fim da linha:

1. **treinar** um `RandomForestRegressor` para prever o `preco` do anúncio;
2. **rastrear** os experimentos no **Vertex AI Experiments** (dois runs, comparados lado a lado);
3. **versionar** o modelo no **Vertex AI Model Registry** (v1 e v2, com alias);
4. **servir** o modelo num **Endpoint** e fazer uma **predição online**.

O fio condutor é sempre o mesmo vetor de features. Guarde esta ordem, porque ela reaparece em cada
predição até o fim do notebook:

> **ORDEM CANÔNICA DO VETOR DE FEATURES**
> `[area_util, area_total, quartos, banheiros, garagens]`

> **Pré-requisitos**: ter a tabela gold no dataset `aula_pdm` (região `us-central1`), a API
> `aiplatform.googleapis.com` habilitada e o bucket de artefatos criado — tudo isso está em
> `../01-setup-gcp.md`.

> **Aviso de consumo de créditos**: a turma usa **créditos educacionais** — não há cobrança no cartão
> de ninguém. As seções 0 a 7 praticamente não consomem crédito. A **seção 8 (deploy em Endpoint)
> consome por node-hora, 24/7, mesmo sem tráfego**. Não encerre a aula sem rodar o checklist de
> `../05-encerramento-custos.md`.

> **Tempo**: o deploy da seção 8 leva de **10 a 20 minutos**. Se a aula estiver corrida, dispare a
> seção 8 assim que a seção 7 terminar e volte para discutir Experiments e Registry enquanto provisiona.


## 0. Configuração

Duas coisas acontecem aqui: instalamos as bibliotecas e fixamos os nomes que o notebook inteiro usa.

O `scikit-learn` está **pinado em `1.6.*`** de propósito. O container que vai servir o modelo é o
`sklearn-cpu.1-6`, e um modelo treinado numa versão e desserializado em outra costuma quebrar no
`predict`. Treinar e servir com a mesma versão é o jeito barato de evitar esse problema.


In [ ]:
%pip install -q google-cloud-aiplatform "scikit-learn==1.6.*" joblib db-dtypes


Depois do `%pip install`, **reinicie o kernel** (no BigQuery Studio: menu do runtime → *Restart*) e siga
a partir da próxima célula. É a forma de garantir que o Python carregue a versão do scikit-learn que
acabamos de instalar, e não a que já estava em memória.


In [ ]:
import google.auth

# O projeto vem da credencial do ambiente, como nos notebooks das aulas anteriores.
_, PROJECT_ID = google.auth.default()
REGION = "us-central1"

# Bucket de artefatos da aula (criado em ../01-setup-gcp.md).
BUCKET = f"{PROJECT_ID}-mlops-aula"
DIRETORIO_MODELO = f"gs://{BUCKET}/models/rf/"   # o Registry aponta para o DIRETÓRIO, não para o arquivo
CAMINHO_MODELO = f"{DIRETORIO_MODELO}model.joblib"

# AJUSTE AQUI: o nome da sua tabela gold. O esquema não é fixo — cada turma construiu a sua.
GOLD_TABLE = f"{PROJECT_ID}.aula_pdm.imoveis_gold"

# Nomes dos recursos do Vertex AI. Mantenha-os: os scripts e os docs da aula usam exatamente estes.
EXPERIMENT = "preco-imoveis-rf"
MODEL_DISPLAY_NAME = "rf-preco-imoveis"
ENDPOINT_DISPLAY_NAME = "rf-preco-imoveis-endpoint"
SERVING_CONTAINER = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-6:latest"

print("projeto:", PROJECT_ID)
print("regiao :", REGION)
print("gold   :", GOLD_TABLE)
print("modelo :", CAMINHO_MODELO)


In [ ]:
# ORDEM CANÔNICA DO VETOR DE FEATURES — [area_util, area_total, quartos, banheiros, garagens]
# Esta lista é o contrato de entrada do modelo. A mesma ordem vale para o treino, para o
# joblib salvo, para o payload do /predict e para o request.json do gcloud.
COLUNAS_FEATURES = ["area_util", "area_total", "quartos", "banheiros", "garagens"]

ALVO = "preco"

# AJUSTE AQUI: a coluna que identifica o imóvel na sua gold (usada para deduplicar).
CHAVE_IMOVEL = "id"

print("ordem canônica:", COLUNAS_FEATURES)


In [ ]:
from google.cloud import storage

# O bucket deveria vir do setup (../01-setup-gcp.md). Esta célula é idempotente: cria só se faltar.
cliente_gcs = storage.Client(project=PROJECT_ID)
bucket = cliente_gcs.bucket(BUCKET)

if bucket.exists():
    print(f"bucket gs://{BUCKET} já existe")
else:
    bucket = cliente_gcs.create_bucket(BUCKET, location=REGION)
    print(f"bucket gs://{BUCKET} criado em {REGION}")


## 1. Ler a camada gold

Nada de novo aqui: é o mesmo `bigquery.Client` das aulas anteriores. O que muda é o destino — em vez de
gravar uma tabela, o resultado vira um `DataFrame` em memória para o treino.

Trazemos também `bairro` e `cidade`. Elas **não** entram no modelo que vai para produção (a explicação
está na seção 4), mas aparecem na célula opcional da seção 5.2.


In [ ]:
from google.cloud import bigquery

cliente_bq = bigquery.Client(project=PROJECT_ID)

# AJUSTE AQUI se sua gold tiver outros nomes de coluna.
sql = f"""
SELECT
  {CHAVE_IMOVEL},
  area_util,
  area_total,
  quartos,
  banheiros,
  garagens,
  bairro,
  cidade,
  {ALVO}
FROM `{GOLD_TABLE}`
WHERE {ALVO} IS NOT NULL
  AND {ALVO} > 0
"""

df = cliente_bq.query(sql).to_dataframe()
print(f"{len(df)} linhas lidas de {GOLD_TABLE}")
df.head()


In [ ]:
df.dtypes


In [ ]:
df[COLUNAS_FEATURES + [ALVO]].describe()


## 2. Deduplicar e separar treino / validação / teste

Duas armadilhas moram nesta seção.

**Anúncio repetido.** As tabelas da aula são append-only: o mesmo imóvel pode ter entrado mais de uma
vez. Se uma cópia cair no treino e outra na validação, o modelo "acerta" um imóvel que já viu e o MAE
mente para você. Deduplicamos **antes** de dividir.

**Vazamento do alvo.** Qualquer coluna derivada do preço (`preco_fmt`, `preco_por_m2`, faixa de preço)
entrega a resposta ao modelo. O resultado fica ótimo no notebook e inútil na vida real. Por isso a lista
`COLUNAS_FEATURES` é explícita: nada entra sem passar por ela.

A divisão é 70 / 15 / 15 com `random_state=42` — todo mundo na sala chega no mesmo split.


In [ ]:
from sklearn.model_selection import train_test_split

# 1) uma linha por imóvel
df_unico = df.drop_duplicates(subset=[CHAVE_IMOVEL], keep="last").reset_index(drop=True)
print(f"{len(df)} linhas -> {len(df_unico)} imóveis únicos")

# 2) só as features do contrato entram em X. Nada derivado do preço.
X = df_unico[COLUNAS_FEATURES]
y = df_unico[ALVO].astype(float)

# 3) 70 / 15 / 15, reprodutível
X_treino, X_resto, y_treino, y_resto = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_teste, y_val, y_teste = train_test_split(X_resto, y_resto, test_size=0.50, random_state=42)

print(f"treino    : {len(X_treino)}")
print(f"validação : {len(X_val)}")
print(f"teste     : {len(X_teste)}")


## 3. A régua: baseline da mediana

Antes de treinar qualquer coisa, precisamos de um número contra o qual comparar. O baseline mais honesto
aqui é **chutar sempre a mediana do preço de treino**.

A métrica é o **MAE** (erro absoluto médio): em média, quantos reais o modelo erra por anúncio. É fácil
de explicar para quem não é da área — e é o que vamos registrar no Experiments.

Um modelo que não bate a mediana não tem motivo para existir.


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error

mediana_treino = float(y_treino.median())
predicao_baseline = np.full(len(y_val), mediana_treino)

mae_baseline = float(mean_absolute_error(y_val, predicao_baseline))

print(f"mediana do treino : R$ {mediana_treino:,.2f}")
print(f"MAE do baseline   : R$ {mae_baseline:,.2f}")


## 4. O pipeline que vai para produção

Este é o ponto do notebook em que a decisão de MLOps aparece com todas as letras.

O container pré-construído do Vertex AI recebe o payload do `/predict` como uma **lista posicional** —
`[120.0, 150.0, 3, 2, 1]`, sem nomes de coluna — e desserializa o `model.joblib` num ambiente Python que
**não conhece este notebook**. Duas consequências práticas:

- **nada de `lambda` nem de `FunctionTransformer` com função definida aqui**: o container não consegue
  importar essas funções e o `joblib.load` falha;
- **nada de transformação que dependa de nome de coluna**: o que chega é um array, e a posição é o
  único identificador.

Por isso o pipeline que registramos usa **só as 5 features numéricas, na ordem canônica**, e é feito de
duas peças que o próprio scikit-learn sabe desserializar sozinho:

`SimpleImputer(strategy="median")` → `RandomForestRegressor`

O imputer entra porque a gold tem buracos (nem todo anúncio informa vaga de garagem). Não há
`StandardScaler`: árvore não se importa com escala.

> As categóricas `bairro` e `cidade` podem melhorar o modelo — a seção **5.2** mostra isso como
> exercício. Mas o modelo **registrado e deployado nesta aula é o numérico**, e é ele que respeita o
> contrato de entrada acima.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


def construir_pipeline(n_estimators=200, max_depth=12, random_state=42):
    """Pipeline que será servido pelo container sklearn-cpu.1-6.

    Restrições de serving (não relaxe sem testar o /predict):
      - entrada = array posicional [area_util, area_total, quartos, banheiros, garagens];
      - sem lambda e sem FunctionTransformer com função local (não desserializa no container);
      - só estimadores do próprio scikit-learn.
    """
    return Pipeline(
        steps=[
            ("imputacao", SimpleImputer(strategy="median")),
            (
                "floresta",
                RandomForestRegressor(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    random_state=random_state,
                    # n_jobs=2 e 200 arvores para caber em runtimes pequenos; -1 pode estourar memoria
                    n_jobs=2,
                ),
            ),
        ]
    )


pipeline = construir_pipeline()
pipeline.fit(X_treino, y_treino)

mae_val = float(mean_absolute_error(y_val, pipeline.predict(X_val)))

print(f"MAE do baseline   : R$ {mae_baseline:,.2f}")
print(f"MAE da floresta   : R$ {mae_val:,.2f}")
print(f"ganho sobre o baseline: {100 * (1 - mae_val / mae_baseline):.1f}%")


In [ ]:
# Sanity check do contrato de entrada: o pipeline aceita a lista posicional que o endpoint vai receber.
exemplo = [[120.0, 150.0, 3, 2, 1]]   # [area_util, area_total, quartos, banheiros, garagens]

print("features:", COLUNAS_FEATURES)
print("entrada :", exemplo[0])
print(f"predição: R$ {pipeline.predict(exemplo)[0]:,.2f}")


## 5. Rastrear os treinos no Vertex AI Experiments

Até aqui, os números do treino existem só na saída da célula. Feche o notebook e eles somem — e com eles
a resposta para "por que escolhemos este modelo?".

O **Experiments** resolve isso: cada treino vira um **run** com seus parâmetros e suas métricas, guardado
no projeto e visível no console em **Vertex AI → Experiments**.

Vamos rodar **dois runs** variando a profundidade da árvore e comparar. Três observações:

- registramos **métricas-resumo** (`log_metrics`), que **não** precisam de TensorBoard. Evitamos de
  propósito `log_time_series_metrics`, que exigiria uma instância de TensorBoard paga;
- **não criamos TensorBoard porque passamos `experiment_tensorboard=False` no `aiplatform.init(...)`.**
  Sem esse parâmetro, o SDK **cria sozinho** uma instância *Default Tensorboard* ao associar o
  experimento — e ela cobra por armazenamento. É por isso que o `30_teardown.sh` verifica a lista de
  TensorBoards no encerramento;
- o Experiments **não existe em gcloud nem em Terraform**. É SDK ou console — e só.


In [ ]:
from datetime import datetime

from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    experiment=EXPERIMENT,
    staging_bucket=f"gs://{BUCKET}",
    # experiment_tensorboard=False evita criar uma instancia Default Tensorboard
    # (metricas-resumo nao precisam dela; ver 05-encerramento-custos)
    experiment_tensorboard=False,
)

print(f"experimento '{EXPERIMENT}' pronto em {REGION}")


In [ ]:
def rodar_e_registrar(sufixo, n_estimators, max_depth):
    """Treina, avalia na validação e registra um run no Vertex AI Experiments."""
    pipe = construir_pipeline(n_estimators=n_estimators, max_depth=max_depth)
    pipe.fit(X_treino, y_treino)
    mae = float(mean_absolute_error(y_val, pipe.predict(X_val)))

    nome_run = f"run-{sufixo}-{datetime.now():%Y%m%d-%H%M%S}"

    aiplatform.start_run(nome_run)
    aiplatform.log_params(
        {
            "modelo": "RandomForestRegressor",
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "features": ",".join(COLUNAS_FEATURES),
            "n_treino": len(X_treino),
        }
    )
    aiplatform.log_metrics({"mae": mae, "mae_baseline": mae_baseline})
    aiplatform.end_run()

    print(f"{nome_run}: MAE R$ {mae:,.2f}")
    return nome_run, pipe, mae


In [ ]:
# Os dois runs variam SOMENTE max_depth — a comparação fica legível e o custo de memória, previsível.
# n_jobs=2 e 200 arvores para caber em runtimes pequenos; -1 pode estourar memoria

# Run A — árvore mais rasa
run_a, pipe_a, mae_a = rodar_e_registrar("a", n_estimators=200, max_depth=12)

# Run B — mesma quantidade de árvores, porém mais profundas
run_b, pipe_b, mae_b = rodar_e_registrar("b", n_estimators=200, max_depth=24)


In [ ]:
# A tabela de runs do experimento — o mesmo que o console mostra em Vertex AI -> Experiments.
aiplatform.get_experiment_df(EXPERIMENT)


Abra agora **Vertex AI → Experiments → `preco-imoveis-rf`** no console, marque os dois runs e clique em
**Compare**. É a mesma informação da tabela acima, com a vantagem de ficar disponível para quem não abriu
este notebook. Esse é o ponto: o experimento deixou de ser um número perdido numa saída de célula.


In [ ]:
# O vencedor por MAE segue para o Registry.
if mae_a <= mae_b:
    pipeline_final, mae_final, run_final = pipe_a, mae_a, run_a
else:
    pipeline_final, mae_final, run_final = pipe_b, mae_b, run_b

mae_teste = float(mean_absolute_error(y_teste, pipeline_final.predict(X_teste)))

print(f"run escolhido : {run_final}")
print(f"MAE validação : R$ {mae_final:,.2f}")
print(f"MAE teste     : R$ {mae_teste:,.2f}   <- a estimativa honesta, em dados nunca vistos")


### 5.2. Opcional — além da aula: e se usássemos `bairro` e `cidade`?

Localização costuma explicar muito do preço de um imóvel, e a gold tem essas colunas. A célula abaixo
treina uma variante com `OneHotEncoder` sobre as categóricas e registra o run no mesmo experimento — dá
para comparar no console com os runs A e B.

> **Este modelo não é o que vai para o Registry.** Ele usa um `ColumnTransformer` sobre um `DataFrame`
> com nomes de coluna, e o endpoint recebe um array posicional. Servir esta variante exigiria um
> container customizado ou um pré-processamento do lado do cliente — assunto do dia 2. Rode a célula
> pela curiosidade, mas siga a aula com o modelo numérico.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

COLUNAS_CATEGORICAS = ["bairro", "cidade"]

X_treino_cat = df_unico.loc[X_treino.index, COLUNAS_FEATURES + COLUNAS_CATEGORICAS]
X_val_cat = df_unico.loc[X_val.index, COLUNAS_FEATURES + COLUNAS_CATEGORICAS]

pre = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), COLUNAS_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), COLUNAS_CATEGORICAS),
    ]
)
# n_jobs=2 e 200 arvores para caber em runtimes pequenos; -1 pode estourar memoria
pipe_cat = Pipeline(steps=[("pre", pre), ("floresta", RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=2))])
pipe_cat.fit(X_treino_cat, y_treino)

mae_cat = float(mean_absolute_error(y_val, pipe_cat.predict(X_val_cat)))

aiplatform.start_run(f"run-categoricas-{datetime.now():%Y%m%d-%H%M%S}")
aiplatform.log_params({"modelo": "RandomForestRegressor", "n_estimators": 200, "max_depth": 12, "features": ",".join(COLUNAS_FEATURES + COLUNAS_CATEGORICAS)})
aiplatform.log_metrics({"mae": mae_cat, "mae_baseline": mae_baseline})
aiplatform.end_run()

print(f"MAE com categóricas : R$ {mae_cat:,.2f}")
print(f"MAE só numéricas    : R$ {mae_final:,.2f}")


## 6. Salvar o artefato: `model.joblib` no GCS

O Vertex AI não guarda o objeto Python: ele guarda um **arquivo no Cloud Storage** e, na hora de servir,
monta esse arquivo dentro do container.

Duas regras que costumam custar meia hora de depuração:

- o arquivo **precisa se chamar `model.joblib`** (não `modelo.pkl`, não `model.pkl`) — é o nome que o
  container sklearn procura;
- o `artifact_uri` do registro aponta para o **diretório** (`.../models/rf/`), nunca para o arquivo.


In [ ]:
import joblib
import sklearn

print("scikit-learn:", sklearn.__version__, "(o container é o sklearn-cpu.1-6)")

joblib.dump(pipeline_final, "model.joblib")

blob = cliente_gcs.bucket(BUCKET).blob("models/rf/model.joblib")
blob.upload_from_filename("model.joblib")

print("artefato em:", CAMINHO_MODELO)
print("diretório registrado:", DIRETORIO_MODELO)


## 7. Registrar no Model Registry (v1 e v2)

O **Model Registry** é o catálogo de modelos do projeto: quem treinou, com qual artefato, com qual
container, em qual versão. **Registrar é grátis** — paga-se o storage do artefato (centavos) e, adiante,
o endpoint.

Fazemos duas coisas aqui:

1. **v1** — `Model.upload(...)` cria a entrada no Registry;
2. **v2** — o mesmo `upload`, agora com `parent_model=<id da v1>`, cria uma **nova versão** do mesmo
   modelo em vez de um modelo novo. É assim que o histórico fica junto.

Aliases (`version_aliases`) dão nome de negócio às versões — `campeao`, `producao` — para o endpoint
apontar para um nome estável em vez de um número. O alias `default` é reservado e acompanha a versão
padrão, definida por `is_default_version=True`.

> **Terraform não registra modelos treinados** (não existe `google_vertex_ai_model`). Este passo é SDK,
> gcloud ou console — os três caminhos estão em `../03-model-registry.md`.


In [ ]:
modelo_v1 = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    artifact_uri=DIRETORIO_MODELO,
    serving_container_image_uri=SERVING_CONTAINER,
    description="RandomForest para preço de anúncio a partir da gold de imóveis.",
    labels={"aula": "mlops", "framework": "sklearn"},
    sync=True,
)

print("resource_name:", modelo_v1.resource_name)
print("versão       :", modelo_v1.version_id)


In [ ]:
# Imagine que o time retreinou com mais dados. O artefato novo vira a VERSÃO 2 do MESMO modelo —
# e não um segundo modelo solto no catálogo. É o parent_model que faz essa amarração.
modelo_v2 = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    parent_model=modelo_v1.resource_name,
    artifact_uri=DIRETORIO_MODELO,
    serving_container_image_uri=SERVING_CONTAINER,
    version_description=f"Retreino do run {run_final} — MAE validação R$ {mae_final:,.2f}.",
    is_default_version=True,
    version_aliases=["campeao"],
    sync=True,
)

print("resource_name:", modelo_v2.resource_name)
print("versão       :", modelo_v2.version_id)
print("aliases      :", modelo_v2.version_aliases)


In [ ]:
# O CATÁLOGO, como o console mostra em Vertex AI -> Model Registry: UMA linha por modelo,
# sempre na versão DEFAULT. Model.list() não devolve o histórico de versões.
for m in aiplatform.Model.list(filter=f'display_name="{MODEL_DISPLAY_NAME}"'):
    print(m.version_id, m.resource_name, m.version_aliases)


In [ ]:
# TODAS as versões do modelo — o histórico que o Model.list() acima NÃO mostra.
# Equivalente em linha de comando: gcloud ai models list-version <MODEL_ID> --region=us-central1
for v in aiplatform.Model(modelo_v1.resource_name).versioning_registry.list_versions():
    print(v.version_id, v.version_aliases, v.version_description)

## 8. Deploy em Endpoint e predição online

Registrar é catalogar; **deployar é ligar a máquina**. O endpoint sobe uma réplica do container com o
modelo dentro e passa a responder HTTP.

> ⏱️ **Este passo leva de 10 a 20 minutos.** Dispare a célula e volte para discutir Experiments e
> Registry enquanto provisiona.

> 💸 **A partir daqui o consumo de créditos começa.** Ninguém será cobrado no cartão — os créditos são
> educacionais —, mas o endpoint consome **por node-hora, 24 horas por dia, mesmo sem nenhuma
> requisição**. `min_replica_count=1` significa uma máquina `n1-standard-2` ligada até você
> desligá-la. O checklist de `../05-encerramento-custos.md` existe por causa desta célula.


In [ ]:
# Idempotente: reaproveita o endpoint se ele já existir (evita duplicar custo em reexecuções).
endpoints = aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_DISPLAY_NAME}"')

if endpoints:
    endpoint = endpoints[0]
    print("endpoint reaproveitado:", endpoint.resource_name)
else:
    endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_DISPLAY_NAME)
    print("endpoint criado:", endpoint.resource_name)


In [ ]:
# 10 a 20 minutos. A célula fica ocupada até o modelo ficar servido.
modelo_v2.deploy(
    endpoint=endpoint,
    deployed_model_display_name=MODEL_DISPLAY_NAME,
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=100,
    sync=True,
)

print("modelos servidos neste endpoint:")
for dm in endpoint.list_models():
    print(" -", dm.id, dm.display_name, dm.model)


In [ ]:
# A predição online. O payload é uma LISTA DE LISTAS na ORDEM CANÔNICA:
#   [area_util, area_total, quartos, banheiros, garagens]
resposta = endpoint.predict(instances=[[120.0, 150.0, 3, 2, 1]])

print("features :", COLUNAS_FEATURES)
print("instância:", [120.0, 150.0, 3, 2, 1])
print(f"predição : R$ {resposta.predictions[0]:,.2f}")


## 9. O teste de fechamento: JSON de exemplo e versão servida

A célula abaixo é a que você mostra para alguém de fora: um **JSON igual ao que um cliente HTTP mandaria**
(é exatamente o conteúdo do `request.json` usado com `gcloud ai endpoints predict`, documentado em
`../04-deploy-endpoint.md`), a predição de volta e **qual versão do modelo respondeu**.

Saber a versão que respondeu é metade do valor do Registry: sem isso, quando o número mudar amanhã,
ninguém sabe dizer se mudou o modelo ou mudou o mundo.


In [ ]:
import json

# request.json — a ordem dos números é o contrato: [area_util, area_total, quartos, banheiros, garagens]
requisicao = {
    "instances": [
        [120.0, 150.0, 3, 2, 1],   # apartamento de 120 m², 3 quartos, 2 banheiros, 1 vaga
        [45.0, 55.0, 1, 1, 0],     # kitnet de 45 m², 1 quarto, sem vaga
    ]
}

print(json.dumps(requisicao, indent=2))

resposta = endpoint.predict(instances=requisicao["instances"])

for entrada, previsto in zip(requisicao["instances"], resposta.predictions):
    print(f"{entrada} -> R$ {previsto:,.2f}")

print()
print("modelo servido      :", resposta.model_resource_name)
print("versão do modelo    :", resposta.model_version_id)
print("deployed_model_id   :", resposta.deployed_model_id)


## ⚠️ Encerramento — não feche o notebook antes disto

O endpoint da seção 8 **continua consumindo créditos por node-hora depois que você fechar esta aba**. Ele
não desliga sozinho, não tem timeout de inatividade e não avisa.

O checklist completo, com a ordem correta e os equivalentes no console e no gcloud, está em
**[`../05-encerramento-custos.md`](../05-encerramento-custos.md)**. A ordem importa: **não dá para
deletar um modelo que ainda está deployado**.

1. `undeploy` do modelo no endpoint → deletar o endpoint (é o item que mais consome);
2. deletar as versões do modelo no Registry;
3. remover os artefatos do GCS criados nesta aula;
4. parar o runtime do notebook (BigQuery Studio / Colab Enterprise).

As células abaixo estão **comentadas de propósito** — para você não apagar o endpoint no meio da
demonstração. Descomente e rode ao final da aula.


In [ ]:
# LIMPEZA 1 — undeploy e deleção do endpoint (o item mais caro; faça primeiro)
# endpoint.undeploy_all()
# endpoint.delete()
# print("endpoint removido")


In [ ]:
# LIMPEZA 2 — deletar as versões do modelo no Registry
# (o delete falha enquanto houver versão deployada: rode a LIMPEZA 1 antes)
# for m in aiplatform.Model.list(filter=f'display_name="{MODEL_DISPLAY_NAME}"'):
#     print("deletando", m.resource_name, "versão", m.version_id)
#     m.delete()


In [ ]:
# LIMPEZA 3 — remover os artefatos desta aula no GCS.
# Apaga só o prefixo models/rf/ — o bucket e o dataset são compartilhados com as outras aulas.
# for b in cliente_gcs.list_blobs(BUCKET, prefix="models/rf/"):
#     print("removendo", b.name)
#     b.delete()


**Falta um passo que não dá para fazer daqui**: parar o *runtime* do notebook. No BigQuery Studio, use o
menu do runtime → *Stop*. Ele tem auto-desligamento por inatividade (~180 min), mas até lá está contando.

Depois de rodar as três limpezas, confira no console — **Vertex AI → Online prediction → Endpoints** e
**Vertex AI → Model Registry** devem estar vazios para esta aula. Os runs do Experiments podem ficar:
eles não custam nada e são o registro do que vocês fizeram.
